In [75]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import urllib.parse
import time
import base64
import os
from bs4 import BeautifulSoup
import json
import pandas as pd

# urllib.parse.unquote

def setup_headless_browser():
    """Set up Chrome in headless mode"""
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # Run in background
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [77]:
login_url="https://authentification.bnf.fr/login"
username = "thom.petitjean@hotmail.fr"
password = "Presse écrite BNF 2024"
target_page = "https://www.google.com/url?q=https%3A%2F%2Fbnf.idm.oclc.org%2Flogin%3Furl%3Dhttps%3A%2F%2Fnouveau.europresse.com%2Faccess%2Fip%2Fdefault.aspx%3Fun%3DD000067U_1&sa=D&sntz=1&usg=AOvVaw359KkJUvjTjlJuRfT-OlnE"
bis_target_path = 'https://bnf.idm.oclc.org/login?url=https://nouveau.europresse.com/access/ip/default.aspx?un=D000067U_1&sa=D&sntz=1&usg=AOvVaw359KkJUvjTjlJuRfT-OlnE'
target_pages = [target_page, bis_target_path]

In [8]:
driver = None
wait_time=10

In [9]:
def login(driver):
    if driver is not None:
        driver.quit()
    driver = setup_headless_browser()
    driver.get(login_url)

    # Wait for login form to load (customize selectors as needed)
    wait = WebDriverWait(driver, 10)

    # Find and fill username field
    username_field = wait.until(EC.presence_of_element_located((By.ID, 'username')))
    username_field.send_keys(username)

    # Find and fill password field
    password_field = driver.find_element(By.ID, 'password')
    password_field.send_keys(password)

    # Submit login form
    submit_button = driver.find_element(By.XPATH, '//input[@type="submit"]')
    submit_button.click()

    # Wait for login to complete
    wait.until(EC.url_changes(login_url))

    for i, target_page in enumerate(target_pages):
        # Navigate to target page
        driver.get(target_page)
        
        # Optional: Add additional interaction or data extraction here
        
        # Example: Print current page title
        print(f"Current Page Title: {driver.title}")
        driver.save_screenshot(driver.title + f"_{i}.png")
        time.sleep(5)
        driver.save_screenshot(driver.title + f"_sleep_{i}.png")

    return driver

In [96]:
driver = login(driver)

Current Page Title: Europresse
Current Page Title: Europresse


In [98]:
data[0]

{'Code': 'UK_P',
 'Name': '01 net',
 'Description': "Créé en 1998, 01 net s'adresse à un large public. Son langage est simple et vulgarisateur. Il simplifie les nouvelles technologies et les rend accessibles à tous grâce à une approche pédagogique, consumériste et ludique.",
 'NameSort': '01 net',
 'Logo': 'uk3_small.gif',
 'SourceLastEdition': '2024-11-20T00:00:00',
 'SourceISSN': '2266-7989',
 'SourceCountry': 'France',
 'Language': '',
 'SourceArchivedSince': '2013-03-07T00:00:00',
 'SourceFinArchive': None,
 'SourceDomaine': 'Informatique et télécommunication',
 'SourceFrequency': 'Mensuel ou bimensuel',
 'PreviousEdition': '2024-11-06T00:00:00'}

In [99]:
driver.get(search_results_url)
soup = BeautifulSoup(driver.page_source)
dict_from_json = json.loads(soup.find("body").text)

data = dict_from_json["SourceResult"]
data = [next(iter(d["SortedSources"].values())) for d in data]
data = {
    d["Code"]: d for d in data
}

In [101]:
with open("journals.json", "w") as f:
    json.dump(data, f)

In [103]:
with open("journals.json", "r") as f:
    d = json.load(f)

In [ ]:
def get_journals(search_results_url):
    driver.get(search_results_url)
    soup = BeautifulSoup(driver.page_source)
    dict_from_json = json.loads(soup.find("body").text)

    data = dict_from_json["SourceResult"]
    data = [next(iter(d["SortedSources"].values())) for d in data]
    data = {
        d["Code"]: d for d in data
    }
    with open("journals.json", "w") as f:
        json.dump(data, f)

    return data

In [93]:
df.Code[0]

'UK_P'

In [84]:
search_results_url = "https://nouveau-europresse-com.bnf.idm.oclc.org/Pdf/GetInitialResults"

In [89]:
df = get_journals(search_results_url)

In [43]:
class src_change(object):
    def __init__(self, locator, src):
        self.locator = locator
        self.src = src

    def __call__(self, driver):
        element = driver.find_element(*self.locator)
        actual_src = element.get_attribute('src')
        if actual_src != self.src:
            return element
        else:
            return False


In [44]:
# Locate button that opens page list side-panel
panel_btn = WebDriverWait(driver, wait_time).until(
    EC.element_to_be_clickable(
        (By.CLASS_NAME, 'pdf-pages-panel-btn')
    )
)

# Open page list side-panel
panel_btn.click()

# Wait until all page links are visible
pdf_page_spans = WebDriverWait(driver, wait_time).until(
    EC.presence_of_all_elements_located(
        (By.CLASS_NAME, 'pdf-page')
    )
)

print(f"Number of pages: {len(pdf_page_spans)}")

img_src = ""

for page_index, span in enumerate(pdf_page_spans):

    print(f"Getting page {page_index}")
    parent_li = span.find_element(By.XPATH, './ancestor::li')
    parent_li = WebDriverWait(driver, wait_time).until(
        EC.element_to_be_clickable(parent_li)
    )
    parent_li.click()

    driver.save_screenshot(f"images/screenshots/page{page_index}.png")

    images = WebDriverWait(driver, wait_time).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, 'viewer-move')
        )
    )
    assert len(images)==1

    wait = WebDriverWait(driver, 10)
    img = wait.until(
        src_change(
            (By.CLASS_NAME, 'viewer-move'), 
            img_src
        )
    )

    #img = images[0]
    img_src = img.get_attribute('src')
    assert img_src and img_src.startswith('data:image/png;base64,')

    base64_data = img_src.split(',')[1]
    filename = os.path.join("./images/clean/", f'page{page_index}.png')

    # Decode and save the image
    with open(filename, 'wb') as file:
        file.write(base64.b64decode(base64_data))

    if page_index >= 3:
        break

Number of pages: 32
Getting page 0
Getting page 1
Getting page 2
Getting page 3


In [ ]:
def process_pdf_pages(driver, wait_time=10):
# Find all spans with class "pdf-page"


pdf_page_spans = WebDriverWait(driver, wait_time).until(
EC.presence_of_all_elements_located((By.CLASS_NAME, 'pdf-page'))
)

# List to store all image sources
all_image_sources = []
print("PDF", len(pdf_page_spans))
for index, span in enumerate(pdf_page_spans):
print(f"Page {index}")
parent_li = span.find_element(By.XPATH, './ancestor::li')
# Wait for the span to be clickable

try:
    WebDriverWait(driver, wait_time).until(
        EC.element_to_be_clickable(parent_li)
    )
except Exception as e:
    print(e)
#import pdb; pdb.set_trace()
print("try click")
# Click the span
parent_li.click()
print("try wait")
# Wait for potential page/content load after clicking
WebDriverWait(driver, wait_time).until(
    EC.presence_of_element_located((By.TAG_NAME, 'img'))
)
print("try save")
save_page_images(driver, page_index=index)

if index>3:
    break